# Random Forest DDoS Detection Model Training
**Description**: This notebook trains a Random Forest classifier on labeled SDN traffic data to detect DDoS attacks.
**Author**: Lead Developer
**Input**: `../data/processed/labeled_dataset.csv`
**Output**: `../detector.joblib`

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
import time

In [ ]:
# Load the labeled dataset
df = pd.read_csv('../data/processed/labeled_dataset.csv')
df.head()

In [ ]:
# Display class distribution
print("Class Distribution:")
print(df['label'].value_counts())
print("\nPercentage:")
print(df['label'].value_counts(normalize=True) * 100)

# Plot bar chart
plt.figure(figsize=(6, 4))
sns.countplot(x='label', data=df)
plt.title('Distribution of Normal (0) vs DDoS (1) Traffic')
plt.show()

In [ ]:
# Feature selection
features = ['pps', 'bps', 'duration_sec', 'packet_count']
X = df[features]
y = df['label']
print(f"Selected features: {features}")
print(f"X shape: {X.shape}, y shape: {y.shape}")

In [ ]:
# Train/test split (80/20, stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Training set size: {X_train.shape[0]}")
print(f"Testing set size: {X_test.shape[0]}")

In [ ]:
# Train RandomForestClassifier
print("Training Random Forest model...")
clf = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
clf.fit(X_train, y_train)
print("Training complete.")

In [ ]:
# Evaluate
y_pred = clf.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"Accuracy: {acc:.4f}\n")
print("Classification Report:")
print(classification_report(y_test, y_pred))
print("Confusion Matrix:")
cm = confusion_matrix(y_test, y_pred)
print(cm)

In [ ]:
# Plot confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Normal', 'DDoS'], yticklabels=['Normal', 'DDoS'])
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.title('Confusion Matrix Heatmap')
plt.show()

In [ ]:
# Measure inference time
print("Measuring inference time...")
sample_data = X_test.iloc[:1000]
start_time = time.time()
predictions = clf.predict(sample_data)
end_time = time.time()

total_time = end_time - start_time
time_per_prediction_ms = (total_time / 1000) * 1000
print(f"Total time for 1000 predictions: {total_time:.4f} seconds")
print(f"Average time per prediction: {time_per_prediction_ms:.4f} ms")

In [ ]:
# Save model
model_path = '../detector.joblib'
joblib.dump(clf, model_path)
print(f"Model saved to {model_path}")

## Final Checklist
- [ ] Accuracy >= 95%
- [ ] FPR < 2%
- [ ] Inference time < 5ms per prediction